In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


 Cell 0: Initial Setup & Data Loading

In [ ]:
print("--- Step 1: Installing dependencies... ---")
!pip install --quiet tifffile numpy opencv-python-headless scikit-image tqdm torch csbdeep onnx onnxruntime

print("\n--- Step 2: Cloning DeepCAD-RT repository... ---")
!rm -rf /content/DeepCAD-RT
!git clone https://github.com/cabooster/DeepCAD-RT.git /content/DeepCAD-RT

print("\n\n✅ All installations and cloning complete. Please RESTART the session now.")

--- Step 1: Installing dependencies... ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 116.9 MB/s eta 0:00:00

--- Step 2: Cloning DeepCAD-RT repository... ---
Cloning into '/content/DeepCAD-RT'...
remote: Enumerating objects: 1853, done.
remote: Counting objects: 100% (507/507), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 1853 (delta 478), reused 350 (delta 350), pack-reused 1346 (from 2)
Receiving objects: 100% (1853/1853), 31.56 MiB | 29.27 MiB/s, done.
Resolving deltas: 100% (1100/1100), done.


✅ All installations and cloning complete. Please RESTART the session now.


In [ ]:
import os
import sys
import shutil
import numpy as np
import tifffile
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import re

%cd /content/DeepCAD-RT/DeepCAD_RT_pytorch

from deepcad.train_collection import training_class
from deepcad.test_collection import testing_class
from deepcad.utils import get_first_filename

print(f"Changed working directory to: {os.getcwd()}")

def pad_to_divisible(image_stack, divisor=16):
    num_frames, h, w = image_stack.shape
    pad_h = (divisor - h % divisor) % divisor
    pad_w = (divisor - w % divisor) % divisor
    if pad_h == 0 and pad_w == 0: return image_stack, (0, 0, 0, 0)
    pad_top, pad_bottom = pad_h // 2, pad_h - (pad_h // 2)
    pad_left, pad_right = pad_w // 2, pad_w - (pad_w // 2)
    print(f"Padding HxW from {h}x{w} to {h+pad_h}x{w+pad_w}.")
    padded_stack = np.pad(image_stack, ((0, 0), (pad_top, pad_bottom), (pad_left, pad_right)), mode='reflect')
    return padded_stack, (pad_top, pad_bottom, pad_left, pad_right)

def crop_padded_stack(padded_stack, spatial_pads):
    if not any(spatial_pads): return padded_stack
    pad_top, pad_bottom, pad_left, pad_right = spatial_pads
    h_new, w_new = padded_stack.shape[1], padded_stack.shape[2]
    crop_y_end = h_new - pad_bottom if pad_bottom > 0 else h_new
    crop_x_end = w_new - pad_right if pad_right > 0 else w_new
    return padded_stack[:, pad_top:crop_y_end, pad_left:crop_x_end]


TRAIN_TIFF_PATH = "/content/drive/MyDrive/DATA_ROOT/exp_data2/Cy3_Best/Cy3_Best.tif"

"""
TEST_TIFF_PATHS = [
    "/content/drive/MyDrive/DATA_ROOT/exp_data2/30_Green/30_Green.tif",
    "/content/drive/MyDrive/DATA_ROOT/exp_data2/29_Green/29_Green.tif",
    "/content/drive/MyDrive/DATA_ROOT/exp_data2/20_Green/20_Green.tif",
    "/content/drive/MyDrive/DATA_ROOT/exp_data2/19_Green/19_Green.tif",
    "/content/drive/MyDrive/DATA_ROOT/exp_data2/Cy3_Best/Cy3_Best.tif"
]
"""
TEST_TIFF_PATHS = [
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_1.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_2.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_3.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_4.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_5.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_6.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_7.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_8.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_9.00.tif",
    "/content/drive/MyDrive/DATA_ROOT/sim_data_paper_v2/sim_Gauss_Poisson_Est_scale_10.00.tif"
]

OUTPUT_DIR = "/content/drive/MyDrive/DATA_ROOT/exp_data3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Loading TRAINING video from: {TRAIN_TIFF_PATH}")
try:
    training_frames = tifffile.imread(TRAIN_TIFF_PATH).astype(np.float32)
    print(f"✅ Successfully loaded training stack with shape: {training_frames.shape}")
except Exception as e:
    print(f"❌ Error loading training TIFF file: {e}")
    training_frames = None

/content/DeepCAD-RT/DeepCAD_RT_pytorch
Changed working directory to: /content/DeepCAD-RT/DeepCAD_RT_pytorch
Loading TRAINING video from: /content/drive/MyDrive/DATA_ROOT/exp_data2/Cy3_Best/Cy3_Best.tif
✅ Successfully loaded training stack with shape: (916, 256, 256)


Cell 2: Train the DeepCAD-RT Model

In [ ]:
import torch
from deepcad.train_collection import training_class
import os
import shutil
import numpy as np
import tifffile

if 'training_frames' in locals() and training_frames is not None:
    print("\n--- Starting DeepCAD-RT Model Training ---")

    if os.path.exists('./deepcad_train_data'): shutil.rmtree('./deepcad_train_data')
    if os.path.exists('./pth'): shutil.rmtree('./pth')
    if os.path.exists('./log'): shutil.rmtree('./log')
    if os.path.exists('./config'): shutil.rmtree('./config')

    DATA_DIR = './deepcad_train_data'
    os.makedirs(DATA_DIR, exist_ok=True)
    tifffile.imwrite(os.path.join(DATA_DIR, 'cy3_best_train.tif'), training_frames)

    train_dict = {
        'GPU_id': '0' if torch.cuda.is_available() else '-1',
        'data_type': str(training_frames.dtype),
        'datasets_path': DATA_DIR,
        'train_datasets_size': 1000,
        'frame_num': len(training_frames),
        'normalize_factor': np.percentile(training_frames, 99.9),
        'denoise_model': 'DeepCAD_RT',
        'loss_func': 'L1', 'n_epochs': 50,
        'patch_x': 64, 'patch_y': 64, 'patch_t': 16,
        'overlap_factor': 0.5
    }

    tc = training_class(train_dict)
    tc.run()

    print("\n--- Moving final model and config to Google Drive ---")
    deepcad_model_dir = os.path.join(OUTPUT_DIR, 'DeepCAD_model')
    os.makedirs(deepcad_model_dir, exist_ok=True)

    try:
        model_file_path, config_file_path = None, None

        all_models = []
        for dirpath, _, filenames in os.walk('./pth'):
            for f in filenames:
                if f.endswith('.pth'):
                    all_models.append(os.path.join(dirpath, f))

        all_models.sort()
        model_file_path = all_models[-1]

        for dirpath, _, filenames in os.walk('./pth'):
            if 'para.yaml' in filenames:
                config_file_path = os.path.join(dirpath, 'para.yaml')
                break

        if model_file_path and config_file_path:
            final_model_name = os.path.basename(model_file_path)
            dest_model_path = os.path.join(deepcad_model_dir, final_model_name)
            dest_config_path = os.path.join(deepcad_model_dir, 'para.yaml')

            shutil.move(model_file_path, dest_model_path)
            shutil.move(config_file_path, dest_config_path)

            print(f"✅ Final model '{final_model_name}' and config saved to: {deepcad_model_dir}")
        else:
            print("❌ Could not find saved model or config file after searching.")

    except Exception as e:
        print(f"❌ An error occurred while moving model files: {e}")
else:
    print("⚠️ Training frames not found. Please run the setup cell (Cell 1) first.")

print("\n--- Moving best model and config to Google Drive ---")
deepcad_model_dir = os.path.join(OUTPUT_DIR, 'DeepCAD_model')
os.makedirs(deepcad_model_dir, exist_ok=True)

try:
    model_file_path, config_file_path = None, None

    for dirpath, _, filenames in os.walk('./pth'):
        if 'model_best.pth' in filenames:
            model_file_path = os.path.join(dirpath, 'model_best.pth')
            break

    for dirpath, _, filenames in os.walk('./pth'):
        if 'para.yaml' in filenames:
            config_file_path = os.path.join(dirpath, 'para.yaml')
            break

    if model_file_path and config_file_path:
        final_model_name = 'model_best.pth'
        dest_model_path = os.path.join(deepcad_model_dir, final_model_name)
        dest_config_path = os.path.join(deepcad_model_dir, 'para.yaml')

        shutil.move(model_file_path, dest_model_path)
        shutil.move(config_file_path, dest_config_path)

        print(f"✅ Best model '{final_model_name}' and config saved to: {deepcad_model_dir}")
    else:
        print("❌ Could not find saved model or config file after searching.")

except Exception as e:
    print(f"❌ An error occurred while moving model files: {e}")


--- Starting DeepCAD-RT Model Training ---
Training parameters -----> 
{'overlap_factor': 0.5, 'datasets_path': './deepcad_train_data', 'n_epochs': 50, 'fmap': 16, 'output_dir': './results', 'pth_dir': './pth', 'onnx_dir': './onnx', 'batch_size': 1, 'patch_t': 16, 'patch_x': 64, 'patch_y': 64, 'gap_y': 32, 'gap_x': 32, 'gap_t': 8, 'lr': 1e-05, 'b1': 0.5, 'b2': 0.999, 'GPU': '0', 'ngpu': 1, 'num_workers': 0, 'scale_factor': 1, 'train_datasets_size': 1000, 'select_img_num': 1000, 'test_datasize': 400, 'visualize_images_per_epoch': False, 'save_test_images_per_epoch': False, 'colab_display': False, 'result_display': ''}
Image list for training -----> 
Total stack number ----->  1
Noise image name ----->  cy3_best_train.tif
Noise image shape ----->  (916, 256, 256)
Using 1 GPU(s) for training -----> 
[Epoch 1/50] [Batch 1029/1029] [Total loss: 318.19, L1 Loss: 17.91, L2 Loss: 618.46] [ETA: 0:06:32] [Time cost: 08 s]      
[Epoch 2/50] [Batch 1029/1029] [Total loss: 226.26, L1 Loss: 15.86,

From training log epoch 6 has the lowest loss so we save E_50_Iter_1029.pth manually to the directory and use it for evaluation.

Cell 3: Denoise Test Videos

In [ ]:
import torch
from deepcad.test_collection import testing_class
from deepcad.utils import get_first_filename
import os
import shutil
import tifffile
import numpy as np


if 'TEST_TIFF_PATHS' in locals():
    print("\n--- Starting DeepCAD-RT Testing ---")

    deepcad_model_dir = os.path.join(OUTPUT_DIR, 'DeepCAD_model')
    try:
        if os.path.exists(deepcad_model_dir) and len(os.listdir(deepcad_model_dir)) > 0:
             print(f"Found trained model directory: {deepcad_model_dir}")
             model_found = True
        else:
            print(f"❌ Trained model directory is empty or not found: {deepcad_model_dir}")
            model_found = False
    except Exception as e:
        print(f"❌ Error finding model. Please train the model first. Error: {e}")
        model_found = False

    if model_found:
        for test_path in TEST_TIFF_PATHS:
            try:
                print(f"\nProcessing: {os.path.basename(test_path)}")
                test_frames = tifffile.imread(test_path).astype(np.uint16)
                base_filename = os.path.splitext(os.path.basename(test_path))[0]
            except Exception as e:
                print(f"  -> ERROR: Could not load {test_path}. Skipping. \n   {e}")
                continue

            if os.path.exists('./deepcad_test_data'): shutil.rmtree('./deepcad_test_data')
            TEST_DATA_DIR = './deepcad_test_data'
            RESULTS_DIR_TEMP = os.path.join(TEST_DATA_DIR, 'results_temp')
            os.makedirs(RESULTS_DIR_TEMP, exist_ok=True)
            tifffile.imwrite(os.path.join(TEST_DATA_DIR, f"{base_filename}.tif"), test_frames)

            test_dict = {
                'GPU_id': '0' if torch.cuda.is_available() else '-1',
                'data_type': str(test_frames.dtype),
                'denoise_model': os.path.basename(deepcad_model_dir),
                'pth_dir': os.path.dirname(deepcad_model_dir),
                'test_datasize': len(test_frames),
                'datasets_path': TEST_DATA_DIR,
                'output_dir': RESULTS_DIR_TEMP,
                'patch_x': 64, 'patch_y': 64, 'patch_t': 16,
                'overlap_factor': 0.5
            }

            print("  -> Denoising frames...")
            tc = testing_class(test_dict)
            tc.run()

            try:
                found_path = None
                for dirpath, _, filenames in os.walk(RESULTS_DIR_TEMP):
                    for f in filenames:
                        if f.endswith('_output.tif'):
                            found_path = os.path.join(dirpath, f)
                            break
                    if found_path:
                        break

                if found_path:
                    output_path = os.path.join(OUTPUT_DIR, f"{base_filename}_denoised_DeepCAD-RT.tif")
                    shutil.move(found_path, output_path)
                    print(f"  -> ✅ Success! Denoised video saved to: {output_path}")
                else:
                    print("  -> ERROR: Could not find the final denoised .tif file in the results folder.")

            except Exception as e:
                print(f"  -> ERROR: An unexpected error occurred while moving the file: {e}")
else:
    print("⚠️ Test file paths not defined. Please run the setup cell (Cell 1) first.")


--- Starting DeepCAD-RT Testing ---
Found trained model directory: /content/drive/MyDrive/DATA_ROOT/exp_data3/DeepCAD_model

Processing: sim_Gauss_Poisson_Est_scale_1.00.tif
  -> Denoising frames...
Testing parameters -----> 
{'overlap_factor': 0.5, 'datasets_path': './deepcad_test_data', 'fmap': 16, 'output_dir': './deepcad_test_data/results_temp', 'pth_dir': '/content/drive/MyDrive/DATA_ROOT/exp_data3', 'batch_size': 1, 'patch_t': 16, 'patch_x': 64, 'patch_y': 64, 'gap_y': 32, 'gap_x': 32, 'gap_t': 8, 'GPU': '0', 'ngpu': 1, 'num_workers': 0, 'scale_factor': 1, 'test_datasize': 916, 'denoise_model': 'DeepCAD_model', 'visualize_images_per_epoch': False, 'colab_display': False, 'result_display': ''}
Stacks for processing -----> 
Total stack number ----->  1
sim_Gauss_Poisson_Est_scale_1.00.tif
Using 1 GPU(s) for testing -----> 
Testing the last model by default:
[Model 1/1, E_06_Iter_1029.pth] [Stack 1/1, sim_Gauss_Poisson_Est_scale_1.00.tif] [Patch 5586/5586] [Time Cost: 21 s] [ETA: 0